## Concept focus — Iterator protocol and generator state

This exercise teaches the machinery hidden behind every `for` loop. Once you can picture `iter()` and `next()` explicitly, generators, lazy pipelines, and coroutine suspension become easier to reason about.

```text
iterable --iter()--> iterator --next()--> value
                              |
                              +--> StopIteration ends the loop

for x in items:
    ...

is secretly:
    it = iter(items)
    while True:
        x = next(it)
```

### How to think about it
Treat an iterator as a moving cursor, not a reusable container. A generator is especially important because it stores execution state between `next()` calls: local variables, the current line, and the pending return path.

### Visual references and further study
- [Python docs — iterator types](https://docs.python.org/3/library/stdtypes.html#iterator-types)
- [PEP 255 — generators](https://peps.python.org/pep-0255/)
- [Python Tutor visualizer](https://pythontutor.com/visualize.html)
- [David Beazley generator talk](https://www.dabeaz.com/generators/)

---

# Module 14 — Iterators, Generators, and Lazy Pipelines

## Exercise 14.1 — The protocol, by hand and by yield

Run:  python ex01_protocol.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. The iterator protocol

Two methods, and everything else in this module is built on them.

In [ ]:
iter(obj)      # -> obj.__iter__()  : returns an ITERATOR
next(it)       # -> it.__next__()   : the next value, or raises StopIteration

A `for` loop is sugar for exactly this:

In [ ]:
for x in things: process(x)

# is precisely:
it = iter(things)
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    process(x)

| | Iterable | Iterator |
|---|---|---|
| Defines | `__iter__` | `__iter__` **and** `__next__` |
| `__iter__` returns | a **fresh** iterator | `self` |
| Reusable | yes | **no** |
| Examples | `list`, `dict`, `str`, `range` | generators, file objects, `iter([])` |

**Every iterator is an iterable; not every iterable is an iterator.** The
distinction shows up as the one-shot bug (Module 09), and it is worth being able
to state precisely:

In [ ]:
data = (x for x in range(3))
list(data)      # [0, 1, 2]
list(data)      # []          <- exhausted, silently

No error. That silence is the whole hazard.

---

## Concept 2. Generator functions

Any function containing `yield` is a generator function. Calling it **runs
nothing** — it returns a generator object.

In [ ]:
def countdown(n: int):
    print("starting")            # does NOT run on the call
    while n > 0:
        yield n
        n -= 1
    print("done")

gen = countdown(3)               # nothing printed
next(gen)                         # 'starting', then 3
next(gen)                         # 2

`yield` **suspends** the function: locals, instruction pointer, and the whole
frame are preserved. `next()` resumes exactly where it stopped. That suspended
frame is the mental image to carry — it is also how `await` works (Module 22).

### Generators are the easiest way to write `__iter__`

Compare this with Module 09's iterator class:

In [ ]:
class Countdown:
    def __init__(self, start: int) -> None:
        self.start = start

    def __iter__(self):
        current = self.start      # a LOCAL, so each call gets fresh state
        while current > 0:
            yield current
            current -= 1

Two `for` loops both work, because each call to `__iter__` creates a new
generator with its own locals. That is the fix for the one-shot bug, and it is
free.

### `yield from`

In [ ]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)      # delegate, recursively
        else:
            yield item

`yield from x` is not just `for i in x: yield i` — it also forwards `send`,
`throw` and `close`, and propagates the sub-generator's return value. For plain
iteration the loop is equivalent; for coroutines it is not.

### Generator expressions

In [ ]:
squares = (x * x for x in range(1_000_000))     # lazy, ~200 bytes
squares = [x * x for x in range(1_000_000)]     # eager, ~40 MB

sum(x * x for x in data)                         # parens optional as sole arg
any(line.startswith("ERROR") for line in fh)     # short-circuits

**Use a generator expression when the values are consumed once.** Use a list
when you need to index, re-iterate, or take `len()`.

---

## Concept 5. Generators as coroutines

`yield` is an expression, so a generator can *receive* values.

In [ ]:
def averager():
    total, count = 0.0, 0
    average = None
    while True:
        value = yield average        # RECEIVES from send(), yields the average
        total += value
        count += 1
        average = total / count

avg = averager()
next(avg)              # "prime" it: run to the first yield
avg.send(10)           # 10.0
avg.send(20)           # 15.0

Three methods drive a generator from outside:

In [ ]:
gen.send(value)        # resume, with `value` as the result of the yield
gen.throw(SomeError)   # raise inside the generator at the yield point
gen.close()            # raise GeneratorExit at the yield point

This is where `async`/`await` came from historically — before native
coroutines, `asyncio` was built on `yield from` over generators. You will rarely
write `send()` today, but understanding it makes Module 22 straightforward
rather than mysterious.

`contextlib.contextmanager` is the one place you use this daily:

In [ ]:
@contextmanager
def managed():
    setup()
    try:
        yield resource        # the with-block runs HERE, at the suspension
    finally:
        teardown()

The generator suspends at `yield`, the `with` body runs, and then the generator
is resumed to run its `finally`. `__exit__` is implemented by calling `send` or
`throw` on it — a direct application of everything above.

---

## Concept 6. When *not* to be lazy

Laziness is not free, and it is not always right.

| Situation | Use |
|---|---|
| Need `len()` | a list |
| Need to iterate twice | a list |
| Need indexing or slicing | a list |
| Small data (under ~1000 items) | a list — clearer, and faster |
| Result feeds a C library (NumPy, pandas) | a list or array |
| Data larger than memory | a generator |
| Infinite or unbounded stream | a generator |
| Early termination likely | a generator |
| Expensive per-item work, may not need all | a generator |

**The debugging cost is real.** A generator pipeline shows you nothing until it
runs, a traceback points at the *consumption* site rather than the definition,
and you cannot inspect intermediate state in a debugger without consuming it.
`list()` a stage temporarily when debugging.

**The exhaustion bug is the one that bites.** Passing a generator to a function
that iterates it twice produces an empty second pass and no error at all
(Module 05's exercise). If a function must iterate twice, it should take a
`Sequence`, not an `Iterable`, and say so in its signature.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The iterator protocol
- Section 2: Generator functions
- Section 3: Pipelines
- Section 4: `itertools`
- Section 5: Generators as coroutines
- Section 6: When *not* to be lazy

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from collections.abc import Iterator


# TODO 1: implement the protocol BY HAND, correctly ---------------------------

---

## `Fibonacci`

Yield the first n Fibonacci numbers.

In [ ]:
class Fibonacci:
    """Yield the first n Fibonacci numbers.

    Write TWO classes: Fibonacci (the iterable) and FibonacciIterator (the
    cursor). Fibonacci.__iter__ must return a FRESH cursor each call.

    Then write BrokenFibonacci, whose __iter__ returns self, and demonstrate
    with an assertion that two consecutive for loops over it disagree.
    """

---

## `FibonacciGen`

One class, one method, three lines. Two loops both work, because each

In [ ]:
class FibonacciGen:
    """One class, one method, three lines. Two loops both work, because each
    call to __iter__ creates a new generator with its own locals."""

---

## `Countdown`

Print something in __iter__ AND in the body, then show:

In [ ]:
class Countdown:
    """Print something in __iter__ AND in the body, then show:
      - creating the object prints nothing
      - calling iter() prints nothing (why not? think about what a generator
        function call does)
      - the first next() prints both
    """

---

## `primes`

An infinite generator of primes. Must work with islice and takewhile,

In [ ]:
def primes() -> Iterator[int]:
    """An infinite generator of primes. Must work with islice and takewhile,
    and must not precompute a bound."""
    raise NotImplementedError

---

## `average_and_max`

Currently iterates its argument TWICE. Given a generator, the second

In [ ]:
def average_and_max(numbers) -> tuple[float, float]:  # type: ignore[no-untyped-def]
    """Currently iterates its argument TWICE. Given a generator, the second
    pass sees nothing and max() raises on an empty sequence.

    Fix it THREE ways and say when each is right:
      (a) materialise with list() at the top
      (b) one pass, tracking both values
      (c) change the signature to Sequence and let the type checker enforce it
    """
    total = sum(numbers)
    count = sum(1 for _ in numbers)          # already empty
    return total / count, max(numbers)

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    fib = FibonacciGen(10)                                # type: ignore[name-defined]
    assert list(fib) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
    assert list(fib) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34], "must be reusable"

    from itertools import islice, takewhile
    assert list(islice(primes(), 5)) == [2, 3, 5, 7, 11]
    assert list(takewhile(lambda p: p < 20, primes())) == [2, 3, 5, 7, 11, 13, 17, 19]

    assert average_and_max([1, 2, 3]) == (2.0, 3)
    assert average_and_max(x for x in [1, 2, 3]) == (2.0, 3), (
        "must work on a one-shot iterable"
    )
    print("all protocol checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.